# Advanced Dictionary of Big Step Semantics

$$\newcommand\eval{\mathbf{eval}}$$
$$\newcommand\semRule[3]{\begin{array}{c} #1 \\ \hline #2 \\ \end{array}\;(\text{#3}) }$$
$$\newcommand\typeOf{\mathbf{typeOf}}$$

Remember: \
$ \mathbb{R} $ = real numbers \
$ \mathbb{B} $ = booleans \
$ \mathbb{C} $ = closures (these represent functions) \
$ \mathbb{P} $ = references (the P stands for "pointers") \
$ \textbf{error} $ = errors

$$\begin{array}{c}
\text{premises that must hold} \\
\hline
\text{conclusions that can be drawn} \\
\end{array}$$

$ \sigma $ = environment \
$ \text{e} $ = some expression ($ \text{e1} $, $ \text{e2} $) \
$ \text{s} $ = some string \
$ \text{c} $ = some contant (real number) \
$ \in $ = "is contained within"

Our operational semantics defines an __eval__ function that has three parts to it.
$$\eval(\texttt{expr}, \texttt{env}, \texttt{store})  = (\text{value}, \text{new-store}) $$

Let us explain how it will be organized. There are two kinds of bindings: 
- Immutable `val`s are bound to values in the environment.
- Mutable `var`s are bound to `Reference(j)` where `j` is an address in the store.

<hr>

### Constants
Largely unchanged from basic rule.

$$\semRule{}{\eval(\texttt{Const(f)}, \sigma, s) = ( f, s) }{const}$$


<hr>

### Identifiers

For mutable values (`var`), we we automatically chase the value corresponding to the reference in the store and return that.

$$\semRule{x \in \text{domain}(\sigma),\ \sigma(x) = \texttt{Reference}(j), \ \texttt{lookupCell}(s, j) = v}{ \eval(\texttt{Ident(x)}, \sigma, s) = (v, s) }{ident-var-ok}$$

- The variable $x$ belongs to the environment $\sigma$.
- It evalutes to a reference to cell $j$ in store.
- Cell $j$ in the store $s$ has value $v$.
- Evaluating the identifier $x$ under environment $\sigma$ and store $s$ has value $v$.

Reminder: `var` values use the store via environment values being pointers to a location in the store. So, the store is fluid where the environment is not. This allows for __dynamic scoping__.

For an immutable `val`, where the value inside the environment is not a pointer, we have the usual semantics:

$$\semRule{x \in \text{domain}(\sigma),\ \sigma(x) = v, v \not= \texttt{Reference}(j) }{ \eval(\texttt{Ident(x)}, \sigma, s) = (v, s) }{ident-val-ok}$$

<hr>

### Let Bindings

##### LetVar

$$\semRule{\eval(\texttt{e1}, \sigma, s) = (v, s_1), \;\; v \not= \mathbf{error},\;\;\; \texttt{createNewCell}(s_1,v) = (j, s_2)\;\;  }{\eval(\texttt{LetVar(x, e1, e2)}, \sigma, s) =  \eval(\texttt{e2}, \sigma[x \mapsto \texttt{Reference}(j)], s_2 }{let-var-ok}$$

- First evaluate `e1` under the environment $\sigma$ and store $s$, results in value $v$ that is not error and store $s_1$.
- Create a new cell in $s_1$, let it be a reference to cell number j and store $s_2$.
- Evaluating `let var x = e1 in e2` is the same as evaluating `e2` under environment $\sigma[x \mapsto \texttt{Reference}(j)]$ and store $s_2$.

##### AssignVar

$$\semRule{x \in \text{domain}(\sigma),\ \sigma(x) = \texttt{Reference}(j),\;\;\eval(\texttt{e}, \sigma, s) = (v, s_1), \;\;  \texttt{assignToCell}(s_1, j, v) = s_2}{ \eval(\texttt{AssignVar(x, e)}, \sigma, s) = (v, s_2) } {assign-var-ok} $$

- $x$ must be mapped to a reference to cell $j$ in the current environment $\sigma$.
- $e$ must evaluate under $\sigma$ and store $s$ to $v$ with new store $s_1$.
- The store $s_2$ is obtained when we assign the value $v$ to cell $j$ in $s_1$.
- The expression $\texttt{AssignVar}(x, e)$ under env. $\sigma$ and store $s$ yields value $v$ and the store $s_2$.

<hr>

### Subtraction

$$\begin{array}{c}
\textbf{eval}(\texttt{e1}, \sigma) = c_1,\ \textbf{eval}(\texttt{e2}, \sigma) =  c_2 \\
\hline
\textbf{eval}\left(\texttt{Minus(e1, e2)}, \sigma\right) = (c_1 - c_2 ) \\
\end{array} (\text{Minus}) $$

<hr>

### Multiplication

$$\begin{array}{c}
\textbf{eval}(\texttt{e1}, \sigma) = c_1,\ \textbf{eval}(\texttt{e2}, \sigma) =  c_2 \\
\hline
\textbf{eval}\left(\texttt{Mult(e1, e2)}, \sigma\right) = (c_1 \times c_2) \\
\end{array} (\text{Mult}) $$

<hr>

### Division

$$\begin{array}{c}
\textbf{eval}(\texttt{e1}, \sigma) = c_1,\ \textbf{eval}(\texttt{e2}, \sigma) =  c_2,\ c_2 \not= 0\\
\hline
\textbf{eval}\left( \texttt{Div(e1, e2)}, \sigma \right) = (\frac{c_1}{c_2} ) \\
\end{array} (\text{Div}) $$

$$\begin{array}{c}
\textbf{eval}(\texttt{e1}, \sigma) =  c_1,\ \textbf{eval}(\texttt{e2}, \sigma) = c_2,\ {c_1 \in \mathbb{R}, c_2 \in \mathbb{R}},\ c_2 = 0\\
\hline
\textbf{eval}\left(\texttt{Div(e1, e2)}, \sigma\right) = \mathbf{error} \\
\end{array} (\text{Div-ERROR}) $$

<hr>

### Logarithms

$$\begin{array}{c}
\textbf{eval}\left( \texttt{e}, \sigma\right) = c,\ c > 0 \\
\hline 
\textbf{eval}(\texttt{Log(e)}, \sigma) = \log(c) \\
\end{array}\ \text{(Log)}$$

$$\begin{array}{c}
\textbf{eval}(\texttt{e}, \sigma) = c,\ {c \in \mathbb{R}},\ {c \leq 0} \\
\hline 
\textbf{eval}(\texttt{Log(e)}, \sigma) =  \mathbf{error} \\
\end{array}\ \text{(Log-ERROR)}$$

<hr>

### Generalized Basic Arithmetic Semantics
##### Plus, Minus, Div, Mult

These rules apply to the set of basic arithmetic operations `Plus`, `Minus`, `Divide`, `Multiply`. Note that the rule refers to $f_T$ for the operator $T$. 
Define these as $f_{Plus}(x,y) = x + y$, $f_{Mult} = x * y$ and $f_{Minus} (x, y) = x - y$. We call these in-built functions.

$$\begin{array}{c}
eval(\texttt{e1}, \sigma) = v_1,\; \; eval(\texttt{e2}, \sigma) = v_2,\ \ v_1 \in \mathbb{R},\ \ v_2 \in \mathbb{R}, \; \; \texttt{T} \in \{ \texttt{Plus, Minus, Mult} \}  \\
\hline
eval(\texttt{T(e1, e2)}, \sigma) = f_T(v_1, v_2) \\
\end{array} \text{(arith-binop-ok-rule)}$$

The `Div` operator needs special handling in exactly the same way we showed in our previous lectures.

$$\begin{array}{c}
eval(\texttt{e1}, \sigma) = v_1,\; \; eval(\texttt{e2}, \sigma) = v_2,\ \ v_1 \in \mathbb{R},\ \ v_2 \in \mathbb{R}, \; \; v_2 \not= 0, \; \; \texttt{T} \in \{ \texttt{Div} \}  \\
\hline
eval(\texttt{T(e1, e2)}, \sigma) = f_T(v_1, v_2) \\
\end{array} \text{(div-binop-ok-rule)}$$

Let's address type mismatching errors, now

$$\begin{array}{c}
eval(\texttt{e1}, \sigma) = v_1,\; \ v_1 \not\in \mathbb{R},\ ; \; \texttt{T} \in \{ \texttt{Plus, Minus, Mult, Div} \}  \\
\hline
eval(\texttt{T(e1, e2)}, \sigma) = \mathbf{error} \\
\end{array} \text{(arith-binop-type-mismatch-rule-1)}\;\;$$
$$\begin{array}{c}
eval(\texttt{e1}, \sigma) = v_1,\; eval(\texttt{e2}, \sigma) = v_2,\; \ v_1 \in \mathbb{R},\; \; v_2 \not\in \mathbb{R},\ ; \; \texttt{T} \in \{ \texttt{Plus, Minus, Mult, Div} \}  \\
\hline
eval(\texttt{T(e1, e2)}, \sigma) = \mathbf{error} \\
\end{array} \text{(arith-binop-type-mismatch-rule-2)}
$$


<hr>

### Generalized Unary Arithmetic Semantics
##### Exp, Sin, Cos, Log

Wherein $f_{Exp}(x) = e^x,\ f_{Sine}(x) = \sin(x), f_{Cosine}(x) = \cos(x)$:

$$\begin{array}{c}
eval(\texttt{e}, \sigma) = v,\;\; v \in \mathbb{R} \; \; \texttt{T} \in \{ \texttt{Exp, Sine, Cosine} \}  \\
\hline
eval(\texttt{T(e)}, \sigma) = f_T(v) \\
\end{array} \text{(arith-unop-ok-rule)}$$

##### Log

Log needs special rule to deal with argument being positive vs. non-positive.

$$\begin{array}{c}
eval(\texttt{e}, \sigma) = v,\;\; v \in \mathbb{R},\ v > 0  \\
\hline
eval(\texttt{Log(e)}, \sigma) = \log(v) \\
\end{array} \text{(log-ok-rule)}\;\;\;$$
$$\begin{array}{c}
eval(\texttt{e}, \sigma) = v,\;\; v \in \mathbb{R},\ v \leq 0  \\
\hline
eval(\texttt{Log(e)}, \sigma) = \mathbf{error} \\
\end{array} \text{(log-nok-rule)}$$

And now our generalized type mismatch error rule:

$$\begin{array}{c}
eval(\texttt{e}, \sigma) = v,\; \ v \not\in \mathbb{R},\ ; \; \texttt{T} \in \{ \texttt{Sine, Cosine, Exp, Log} \}  \\
\hline
eval(\texttt{T(e)}, \sigma) = \mathbf{error} \\
\end{array} \text{(arith-unop-type-mismatch-rule)}$$

<hr>

### Comprehensive Boolean Semantics

##### __Basic Boolean Values__

$$\begin{array}{c} 
\\
\hline
eval(\texttt{True}, \sigma) = true \\
\end{array} \text{(true rule)} \;\;\;\;
\begin{array}{c} 
\\
\hline
eval(\texttt{False}, \sigma) = false \\
\end{array}\text{(false rule)}
$$

##### __And__ *(Can be short circuited.)*

$$\begin{array}{c} 
eval(\texttt{e1}, \sigma) = false\\
\hline
eval(\texttt{And}(e1, e2), \sigma) = false\\
\end{array} \text{(and-arg-1-ok-rule)} \;\;\;\;
\begin{array}{c} 
eval(\texttt{e1}, \sigma) = v_1, \;\; v_1 \not\in \mathbb{B} \\
\hline
eval(\texttt{And}(e1, e2), \sigma) = \mathbf{error}\\
\end{array}\text{(and-arg-1-nok-rule)}
$$

$$\begin{array}{c} 
eval(\texttt{e1}, \sigma) = true, \;\; eval(\texttt{e2}, \sigma) = v_2, \;\; v_2 \in \mathbb{B}\\
\hline
eval(\texttt{And}(e1, e2), \sigma) = v_2\\
\end{array} \text{(and-arg-2-ok-rule)} \;\;\;\;
\begin{array}{c} 
eval(\texttt{e1}, \sigma) = true, \:\: eval(\texttt{e2}, \sigma) = v_2, \;\; v_2 \not\in \mathbb{B} \\
\hline
eval(\texttt{And}(e1, e2), \sigma) = \mathbf{error}\\
\end{array}\text{(and-arg-2-nok-rule)}
$$

##### __Or__

$$\begin{array}{c} 
eval(\texttt{e1}, \sigma) = true, \;\; eval(\texttt{e2}, \sigma) = v_2, \;\; v_2 \in \mathbb{B} \\
\hline
eval(\texttt{Or}(e1, e2), \sigma) = true\\
\end{array} \text{(or-arg-1-ok-rule)} \;\;\;\;
\begin{array}{c} 
eval(\texttt{e1}, \sigma) = v_1, \;\; v_1 \not\in \mathbb{B} \\
\hline
eval(\texttt{Or}(e1, e2), \sigma) = \mathbf{error}\\
\end{array}\text{(or-arg-1-nok-rule)}
$$

$$\begin{array}{c} 
eval(\texttt{e1}, \sigma) = v_1, \;\; v_1 \in \mathbb{B}, \;\; eval(\texttt{e2}, \sigma) = true \\
\hline
eval(\texttt{Or}(e1, e2), \sigma) = true\\
\end{array} \text{(or-arg-2-ok-rule)} \;\;\;\;
\begin{array}{c} 
eval(\texttt{e2}, \sigma) = v_2, \;\; v_2 \not\in \mathbb{B} \\
\hline
eval(\texttt{Or}(e1, e2), \sigma) = \mathbf{error}\\
\end{array}\text{(or-arg-2-nok-rule)}
$$

Note that or-arg-1-nok-rule is a short circuit.

<hr>

### Let Binding

$ \sigma $ = environment

$$\begin{array}{c} 
eval(\texttt{e1}, \sigma) = v_1,\;\; v_1 \not= \mathbf{error}, \;\; eval(\texttt{e2}, \sigma[x \mapsto v_1]) = v_2, \;\; v_2 \not= \mathbf{error}\\
\hline
eval(\texttt{Let(x,e1,e2)}, \sigma) = v_2\\
\end{array} \text{(let-binding-ok)} $$

$$\begin{array}{c} 
eval(\texttt{e1}, \sigma) =  \mathbf{error}\\
\hline
eval(\texttt{Let(x,e1,e2)}, \sigma) = \mathbf{error}\\
\end{array} \text{(let-binding-nok-1)}$$

$$\begin{array}{c} 
eval(\texttt{e1}, \sigma) =  v_1, \;\; v_1 \not= \mathbf{error}, \;\; eval(\texttt{e2}, \sigma[x \mapsto v_1]) = \mathbf{error}\\
\hline
eval(\texttt{Let(x,e1,e2)}, \sigma) = \mathbf{error}\\
\end{array} \text{(let-binding-nok-2)}$$

<hr>

### TopLevel (Program Level)

$\phi$ = initially empty environment

$$\begin{array}{c}
eval(\texttt{e}, \phi) = v \\
\hline
eval(\texttt{TopLevel(e)}, \phi) = v \\
\end{array} \text{(toplevel-eval)} $$

<hr>

### Closures

$ \text{Closure}( x, e, \sigma)$ that represents a closure 
involving the function definition

`function(x) e` with environment $\sigma$.

When calling a function in a closure, we need to pass a value for the parameter `x`.

<hr>

### Functions

Function definitions and calls will interface with the type $ \mathbb{C} $

##### __Function Definitions__

$$ \begin{array}{c}
\\
\hline
\text{eval}(\texttt{FunDef}(x, e), \sigma) = \text{Closure}(x, \texttt{e}, \sigma) \\
\end{array} \text{(fun-def-ok)}$$

##### __Function Calls__

(more in [Week 7 Recitation](./week-6/recitation/week7_solutions.ipynb))

$$ \begin{array}{c}
\text{eval}(\texttt{funExpr}, \sigma) = \text{Closure}(p, \texttt{e}, \pi),\ \text{eval}(\texttt{argExpr}, \sigma) = v_2,\ v_2 \not= \mathbf{error} \\
\hline
\texttt{eval}( \texttt{FunCall(funExpr, argExpr)}, \sigma )  = \texttt{eval}( \texttt{e}, \pi [ p \mapsto v_2 ] )\\
\end{array} \text{(fun-call-ok)}
$$

The rule concerns itself with evaluating a function call `FunCall(funExpr, argExpr)` of a call `funExpr(argExpr)` where
- `funExpr` represents the function we are going to call.
- `argExpr` represents the argument to the call.

First, if `fun_expr` is a function, then it better evaluate to a _closure_.
- This is expressed by the condition $\text{eval}(\texttt{funExpr}, \sigma) = \text{Closure}(p, \texttt{e}, \pi)$ on the top of the bar. This closure has a formal param $p$, 
body $\texttt{e}$ and the environment that was saved at function define time is $\pi$.

Next, we evaluate the argument to the funciton call `arg_expr`. Let this evaluate to $v_2$ where $v_2$ is not an __error__ value.

Finally, we are ready to call the closure. Note that with the arrival of $v_2$ from previous step, we have
everything needed to evaluate a function call. So we can extend the environment $\pi$ by binding 
$p$ the formal parameter to $v_2$. With that done, everything is ready to evaluate the body of the closure
$\texttt{e}$.

We need to add error rules. If the function expression is not a closure:

$$ \begin{array}{c}
\text{eval}(\texttt{funExpr}, \sigma) \not \in \mathbb{C}\\
\hline
\texttt{eval}( \texttt{FunCall(funExpr, argExpr)}, \sigma )  = \mathbf{error}\\
\end{array} \text{(fun-call-nok-1)}
$$

If the argument to the call leads to error:

$$ \begin{array}{c}
\text{eval}(\texttt{funExpr}, \sigma) = \text{Closure}(x, \texttt{e}, \pi),\ \text{eval}(\texttt{argExpr}, \sigma) = \mathbf{error} \\
\hline
\texttt{eval}( \texttt{FunCall(funExpr, argExpr)}, \sigma )  = \mathbf{error}\\
\end{array} \text{(fun-call-nok-2)}
$$

<hr>

### Title



<hr>

### Title



<hr>

### Title



<hr>

### Title



<hr>

### Title

